
<div style="text-align: center; padding: 30px 10px;">

<h1 style="color:#ff7500; font-size: 24px; margin-bottom: 10px;">
МФТИ ФПМИ
</h1>

<h2 style="font-size: 30px; margin-top: 5px;">
Практикум Python - Основной Поток
</h2>

<hr style="width: 60%; border: 1px solid #10069f; margin: 25px auto;">

<h3 style="font-size: 36px;">
13. Async.
</h3>

<p style="margin-top: 20px;">
<strong>Дата:</strong> 28 апреля - 30 апреля 2026 года<br>
</p>

<p style="margin-top: 25px;">
Данный ноутбук является частью серии семинаров по курсу  
<em>«Практикум Python»</em> и предназначен для учебных и образовательных целей.
</p>

</div>

# Асинхронное программирование в Python


Этот ноутбук — подробное и практическое введение в асинхронность.

Мы разберем:
- что такое синхронный и асинхронный подход;
- зачем нужен `asyncio`;
- что такое **корутина** и почему это ключевая идея;
- как работают `await`, `Task`, `gather`;
- типичные ошибки и хорошие практики.

> Ноутбук ориентирован на новичков: много объяснений, короткие примеры, постепенное усложнение.

## Зачем вообще нужна асинхронность?

Представь, что ты студент и у тебя 3 дела:
1. ждать, пока найдется катка в доту,
2. ждать пока загрузится шортс в йутуб,
3. ждать пока вскипит чайник.

Во всех трех случаях процессор **не занят вычислениями** — мы просто ждем.

### Синхронный стиль
Ты делаешь дела строго по очереди: сначала полностью первое, потом второе, потом третье.

### Асинхронный стиль
Ты начинаешь первое дело, и пока оно «ждет» (I/O), переключаешься на другое. Это не магия и не ускорение процессора, это **умное использование времени ожидания**.

Важно: асинхронность особенно полезна для задач с ожиданием: сеть, файлы, базы данных, внешние API.

In [2]:
import asyncio
import inspect
import time
from random import randint


def line():
    print('-' * 60)


def ts() -> str:
    """Короткий timestamp для красивого вывода."""
    return time.strftime('%H:%M:%S')

## Пример 1. Синхронный код (все строго по очереди)

Сейчас мы специально напишем блокирующий вариант через `time.sleep`.

`time.sleep` «замораживает» поток: пока идет сон, ничего другого в этом потоке не делается.

In [3]:
def sync_job(name: str, delay: float) -> str:
    print(f"[{ts()}] start {name}")
    time.sleep(delay)
    print(f"[{ts()}] done  {name}")
    return f"{name}: {delay}s"

start = time.perf_counter()
results = [
    sync_job('A', 1.0),
    sync_job('B', 1.5),
    sync_job('C', 1.0),
]
end = time.perf_counter()

line()
print('Результаты:', results)
print(f'Итоговое время: {end - start:.2f} сек')
print('Ожидание суммируется: примерно 1.0 + 1.5 + 1.0 = 3.5 сек')

[19:52:50] start A
[19:52:51] done  A
[19:52:51] start B
[19:52:52] done  B
[19:52:52] start C
[19:52:53] done  C
------------------------------------------------------------
Результаты: ['A: 1.0s', 'B: 1.5s', 'C: 1.0s']
Итоговое время: 3.51 сек
Ожидание суммируется: примерно 1.0 + 1.5 + 1.0 = 3.5 сек


## Что такое корутина?

**Корутина** — это специальная функция, объявленная как `async def`.

Ключевые идеи:
- вызов `async`-функции **не выполняет ее сразу**, а создает объект корутины;
- выполнение начинается, когда корутину **ожидают** (`await`) или запускают как задачу (`asyncio.create_task`);
- внутри корутины можно писать `await`, чтобы добровольно «уступить управление» event loop.

Именно это делает возможной кооперативную многозадачность в одном потоке.

In [4]:
async def async_job(name: str, delay: float) -> str:
    print(f"[{ts()}] start {name}")
    await asyncio.sleep(delay)  # неблокирующее ожидание
    print(f"[{ts()}] done  {name}")
    return f"{name}: {delay}s"

coro = async_job('demo', 1.0)
print('Тип объекта:', type(coro))
print('Это корутина?', inspect.iscoroutine(coro))
print('Это функция-корутина?', inspect.iscoroutinefunction(async_job))

# Чтобы не оставлять незапущенную корутину:
await coro

Тип объекта: <class 'coroutine'>
Это корутина? True
Это функция-корутина? True
[19:54:07] start demo
[19:54:08] done  demo


'demo: 1.0s'

## `await`: что происходит на самом деле

Когда интерпретатор видит `await something`:
1. текущая корутина приостанавливается;
2. управление возвращается в event loop;
3. loop может запустить другие готовые задачи;
4. когда `something` готово — корутина продолжится с того же места.

Это и есть «асинхронное переключение», но без потоков и без параллельного исполнения Python-байткода на нескольких ядрах.

In [5]:
# Тот же async-код, но запускаем ПОСЛЕДОВАТЕЛЬНО
start = time.perf_counter()

r1 = await async_job('A', 1.0)
r2 = await async_job('B', 1.5)
r3 = await async_job('C', 1.0)

end = time.perf_counter()
line()
print('Результаты:', [r1, r2, r3])
print(f'Итоговое время: {end - start:.2f} сек')
print('Почти как sync: мы ждем каждую корутину по очереди.')

[19:55:12] start A
[19:55:13] done  A
[19:55:13] start B
[19:55:14] done  B
[19:55:14] start C
[19:55:15] done  C
------------------------------------------------------------
Результаты: ['A: 1.0s', 'B: 1.5s', 'C: 1.0s']
Итоговое время: 3.51 сек
Почти как sync: мы ждем каждую корутину по очереди.


## Конкурентный запуск: `asyncio.gather`

Чтобы получить выигрыш, корутины должны быть «в полете» одновременно.

`asyncio.gather(coro1, coro2, ...)`:
- планирует выполнение всех переданных awaitable;
- ждет завершения всех;
- возвращает результаты в том же порядке, что аргументы.

In [6]:
start = time.perf_counter()

results = await asyncio.gather(
    async_job('A', 1.0),
    async_job('B', 1.5),
    async_job('C', 1.0),
)

end = time.perf_counter()
line()
print('Результаты:', results)
print(f'Итоговое время: {end - start:.2f} сек')
print('Обычно близко к max(1.0, 1.5, 1.0) = 1.5 сек, а не к сумме 3.5 сек.')

[19:56:09] start A
[19:56:09] start B
[19:56:09] start C
[19:56:10] done  A
[19:56:10] done  C
[19:56:10] done  B
------------------------------------------------------------
Результаты: ['A: 1.0s', 'B: 1.5s', 'C: 1.0s']
Итоговое время: 1.50 сек
Обычно близко к max(1.0, 1.5, 1.0) = 1.5 сек, а не к сумме 3.5 сек.


## Чуть глубже: жизненный цикл корутины

Состояния, которые полезно знать:
- создана (но не выполнялась),
- выполняется,
- приостановлена на `await`,
- завершена (`return`) или упала с исключением.

Корутина сама по себе — это еще не задача планировщика. Чтобы loop управлял ей независимо, обычно создают `Task`.

In [7]:
async def tiny():
    await asyncio.sleep(0.1)
    return 'ok'

c = tiny()
print('is coroutine:', inspect.iscoroutine(c))
print('initial state:', inspect.getcoroutinestate(c))

result = await c
print('result:', result)
print('final state:', inspect.getcoroutinestate(c))

is coroutine: True
initial state: CORO_CREATED
result: ok
final state: CORO_CLOSED


## `Task`: фоновое выполнение корутины

`Task` — это объект, который регистрируется в event loop и может выполняться параллельно с другими задачами (конкурентно, не обязательно на разных ядрах).

Полезно, когда нужно:
- запустить несколько действий заранее;
- сделать полезную работу, пока они выполняются;
- потом отдельно дождаться результатов.

In [8]:
async def delayed_square(x: int) -> int:
    await asyncio.sleep(0.5)
    return x * x

task1 = asyncio.create_task(delayed_square(3))
task2 = asyncio.create_task(delayed_square(4))

print('Пока задачи считаются, делаем что-то еще...')
await asyncio.sleep(0.2)
print('task1 done?', task1.done())
print('task2 done?', task2.done())

a = await task1
b = await task2
print('Результаты:', a, b)

Пока задачи считаются, делаем что-то еще...
task1 done? False
task2 done? False
Результаты: 9 16


## Исключения в асинхронном коде

Если одна из задач падает, поведение `gather` зависит от параметров:
- по умолчанию (`return_exceptions=False`) исключение пробрасывается наружу;
- при `return_exceptions=True` исключения возвращаются как элементы результата.

В проде второй режим часто удобен для «собрать все результаты и ошибки разом».

In [9]:
async def unstable_job(i: int) -> str:
    await asyncio.sleep(0.2)
    if i % 3 == 0:
        raise ValueError(f'bad job {i}')
    return f'ok job {i}'

print('Вариант 1: исключение прерывает await gather')
try:
    await asyncio.gather(*(unstable_job(i) for i in range(1, 6)))
except Exception as e:
    print('Поймали:', type(e).__name__, e)

line()
print('Вариант 2: возвращаем и результаты, и ошибки')
results = await asyncio.gather(
    *(unstable_job(i) for i in range(1, 6)),
    return_exceptions=True,
)
for idx, item in enumerate(results, start=1):
    print(idx, '->', repr(item))

Вариант 1: исключение прерывает await gather
Поймали: ValueError bad job 3
------------------------------------------------------------
Вариант 2: возвращаем и результаты, и ошибки
1 -> 'ok job 1'
2 -> 'ok job 2'
3 -> ValueError('bad job 3')
4 -> 'ok job 4'
5 -> 'ok job 5'


## Ограничение конкурентности: `asyncio.Semaphore`

Иногда нельзя запускать слишком много задач сразу:
- API имеет rate limit,
- база данных не любит сотни соединений,
- локальные ресурсы ограничены.

`Semaphore(n)` позволяет держать одновременно не более `n` активных операций.

In [10]:
sem = asyncio.Semaphore(3)

async def limited_worker(i: int) -> str:
    async with sem:
        delay = randint(1, 3) / 3
        print(f'[{ts()}] worker {i} start ({delay:.2f}s)')
        await asyncio.sleep(delay)
        print(f'[{ts()}] worker {i} done')
        return f'worker {i} ok'

start = time.perf_counter()
results = await asyncio.gather(*(limited_worker(i) for i in range(1, 11)))
end = time.perf_counter()

line()
print('Готово задач:', len(results))
print(f'Итоговое время: {end - start:.2f} сек')
print('Заметь: одновременно работало не более 3 workers.')

[20:03:33] worker 1 start (0.33s)
[20:03:33] worker 2 start (1.00s)
[20:03:33] worker 3 start (0.33s)
[20:03:33] worker 1 done
[20:03:33] worker 3 done
[20:03:33] worker 4 start (1.00s)
[20:03:33] worker 5 start (1.00s)
[20:03:34] worker 2 done
[20:03:34] worker 6 start (1.00s)
[20:03:34] worker 4 done
[20:03:34] worker 5 done
[20:03:34] worker 7 start (0.33s)
[20:03:34] worker 8 start (0.33s)
[20:03:35] worker 7 done
[20:03:35] worker 8 done
[20:03:35] worker 9 start (1.00s)
[20:03:35] worker 10 start (0.33s)
[20:03:35] worker 6 done
[20:03:35] worker 10 done
[20:03:36] worker 9 done
------------------------------------------------------------
Готово задач: 10
Итоговое время: 2.67 сек
Заметь: одновременно работало не более 3 workers.


## Producer / Consumer на `asyncio.Queue`

Очень частый шаблон: один блок производит данные, другой обрабатывает.

`asyncio.Queue` удобна тем, что:
- producer делает `await queue.put(...)`;
- consumer делает `await queue.get()`;
- при пустой очереди consumer не крутит CPU, а корректно ждет.

In [11]:
queue = asyncio.Queue()

async def producer(n: int):
    for i in range(1, n + 1):
        await asyncio.sleep(0.15)
        item = i * 10
        await queue.put(item)
        print(f'producer -> {item}')
    await queue.put(None)  # сигнал завершения


async def consumer():
    total = 0
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done()
            break
        await asyncio.sleep(0.1)
        total += item
        print(f'consumer <- {item}, total={total}')
        queue.task_done()
    return total

producer_task = asyncio.create_task(producer(5))
consumer_task = asyncio.create_task(consumer())

await producer_task
await queue.join()
result = await consumer_task
print('Финальная сумма:', result)

producer -> 10
consumer <- 10, total=10
producer -> 20
consumer <- 20, total=30
producer -> 30
consumer <- 30, total=60
producer -> 40
consumer <- 40, total=100
producer -> 50
consumer <- 50, total=150
Финальная сумма: 150


## Важное ограничение: async не ускоряет тяжелые вычисления

Асинхронность отлично работает для I/O-bound задач (ожидание сети, диска, БД).

Для CPU-bound задач (долгие вычисления) `asyncio` обычно не дает ускорения само по себе.

Если нужно встроить блокирующую функцию в async-код, можно вынести ее в поток через `asyncio.to_thread`.

In [12]:
def blocking_io(name: str, delay: float) -> str:
    time.sleep(delay)
    return f'{name} finished'

start = time.perf_counter()
results = await asyncio.gather(
    asyncio.to_thread(blocking_io, 'job-1', 1.0),
    asyncio.to_thread(blocking_io, 'job-2', 1.2),
    asyncio.to_thread(blocking_io, 'job-3', 0.8),
)
end = time.perf_counter()

print(results)
print(f'Через to_thread заняло: {end - start:.2f} сек')

['job-1 finished', 'job-2 finished', 'job-3 finished']
Через to_thread заняло: 1.20 сек


## Частые ошибки новичков

1. **Забыли `await`**
   - Вызвали `async_func()`, получили объект корутины, но не запустили.

2. **Используют `time.sleep` внутри `async def`**
   - Это блокирует event loop. Нужен `await asyncio.sleep(...)`.

3. **Думают, что async = многопоточность**
   - Нет, это кооперативная конкуренция в одном потоке (в базовом сценарии).

4. **Запускают тысячи задач без лимитов**
   - Нужны `Semaphore`, пулы или батчинг.

5. **Не обрабатывают исключения задач**
   - Используй `try/except`, `gather(..., return_exceptions=True)` или контроль задач по отдельности.

## Мини-практика

Попробуй сделать сам:

1. Напиши корутину `fetch_fake(i)`, которая ждет случайное время от 0.2 до 1.0 сек и возвращает `i`.
2. Запусти 20 таких корутин через `gather` и замерь общее время.
3. Ограничь конкурентность до 5 через `Semaphore` и сравни время.
4. Добавь ошибку для каждого 4-го `i` и собери результаты с `return_exceptions=True`.

Если всё получилось — ты уже уверенно понимаешь основы асинхронного подхода.

## Краткий итог

- Асинхронность нужна, когда много ожидания I/O.
- Корутина (`async def`) — базовая единица async-кода.
- `await` приостанавливает текущую корутину и дает loop выполнять другие.
- `Task` и `gather` помогают запускать много операций конкурентно.
- Для практики важны контроль ошибок, ограничение конкурентности и аккуратная архитектура.

Дальнейший шаг: применить это к реальному HTTP-клиенту (`aiohttp`) или асинхронной работе с БД.